# Notebook 1: Get the pages ready

This notebook downloads the competition's pages (all three collections) from S3, and prepares them for
training:

- it sets aside the team's **dev pages** (55 pages from whole documents, listed in `dev_ids.csv`),
  which are never trained on and are what every run is scored on;
- it cuts every **survey spread** (an open notebook, two pages side by side) at the fold, so the
  model reads one page at a time, and splits the label to match where it can;
- it takes the `Postmark` line out of the Kade labels it trains on (the postmark is on the back of
  the card, so it isn't in the image).

Run it once. About 15 to 20 minutes, mostly downloading about 2.5 GB of images.
Run the cells from top to bottom with **Shift + Enter**. Kernel: **conda_pytorch_p310** (or another `conda_pytorch` kernel, not `conda_python3`).

## Step 1. Set up the folders

Nothing to change here.

In [ ]:
import importlib.util
import os
import subprocess
import sys

# Which kernel is this? It should be conda_pytorch_p310 (or another conda_pytorch kernel).
print("Kernel environment:", os.path.basename(os.path.dirname(os.path.dirname(sys.executable))))
missing = [pkg for module, pkg in (("PIL", "pillow"), ("pandas", "pandas"))
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing", ", ".join(missing), "into this kernel ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Everything this project downloads or creates lives here, on the notebook
# instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/churro-work"
DATA = os.path.join(WORK, "data")     # page images and lists of pages
RUNS = os.path.join(WORK, "runs")     # one folder per training run
SCRIPTS = os.path.abspath("scripts")  # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")   # the 7.5 GB model download goes here
print("Working folder:", WORK)

## Step 2. Download the pages

This needs the **full competition data**: `train.csv`, `test.csv` and the `images/` folder with
all three collections. The class bucket (`ml-marathon-shared`) only has the Kade letters. Upload
the Kaggle download to your own bucket first (README.md, "Where the data comes from"), then type
its name and folder below.

If the files aren't where you said, the cell searches the bucket and tells you where `train.csv`
is. Images already downloaded are skipped, so it's safe to run again.

In [ ]:
import glob

import boto3
import pandas as pd
from botocore.exceptions import ClientError

DATA_BUCKET = "your-bucket-name"   # <-- EDIT ME: the bucket with the full competition data
DATA_PREFIX = "badger-scribe"      # <-- the folder in it that holds train.csv, test.csv and images/

if DATA_BUCKET == "your-bucket-name":
    raise SystemExit("Type your bucket's name in DATA_BUCKET first (README.md, 'Where the data comes from').")
s3 = boto3.client("s3")
images_dir = os.path.join(DATA, "images")
os.makedirs(images_dir, exist_ok=True)

def find_in_bucket(ending, most=10):
    found = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=DATA_BUCKET):
        found += [o["Key"] for o in page.get("Contents", []) if o["Key"].endswith(ending)]
        if len(found) >= most:
            break
    return found[:most]

def explain(error, key):
    code = error.response.get("Error", {}).get("Code", "")
    if code in ("404", "NoSuchKey", "NotFound"):
        try:
            hits = find_in_bucket("train.csv")
        except ClientError:
            hits = None
        if hits:
            folders = sorted({h.rsplit("/", 1)[0] if "/" in h else "" for h in hits})
            return (f"There is no {key} in {DATA_BUCKET}. train.csv is in: " + ", ".join(repr(f) for f in folders)
                    + ". Set DATA_PREFIX to one of these and run again.")
        if hits == []:
            return f"The bucket {DATA_BUCKET} has no train.csv anywhere. Upload the data first (README.md)."
        return f"There is no {key} in {DATA_BUCKET}, and this notebook isn't allowed to look inside it."
    if code in ("403", "AccessDenied"):
        return f"This notebook isn't allowed to read {DATA_BUCKET}. Check the bucket's permissions (README.md)."
    if code == "NoSuchBucket":
        return f"There is no bucket called {DATA_BUCKET}. Check the name in the S3 console."
    return f"S3 said: {error}"

prefix = f"{DATA_PREFIX}/" if DATA_PREFIX else ""
for name in ("train.csv", "test.csv"):
    try:
        s3.download_file(DATA_BUCKET, prefix + name, os.path.join(DATA, name))
    except ClientError as e:
        raise SystemExit(explain(e, prefix + name))
for name in ("metadata.csv", "sample_submission.csv"):   # useful, but not required
    try:
        s3.download_file(DATA_BUCKET, prefix + name, os.path.join(DATA, name))
    except ClientError:
        print(f"(no {name}; that's fine)")

train = pd.read_csv(os.path.join(DATA, "train.csv"), keep_default_na=False, encoding="utf-8-sig")
test = pd.read_csv(os.path.join(DATA, "test.csv"), keep_default_na=False, encoding="utf-8-sig")
wanted = list(pd.concat([train, test])["page_id"].drop_duplicates())
new, missing = 0, []
for i, page_id in enumerate(wanted, 1):
    path = os.path.join(images_dir, page_id + ".jpg")
    if not os.path.exists(path):
        try:
            s3.download_file(DATA_BUCKET, f"{prefix}images/{page_id}.jpg", path)
            new += 1
        except ClientError:
            missing.append(page_id)
    if i % 100 == 0:
        print(f"  {i}/{len(wanted)} pages checked", flush=True)
print(f"{len(train)} training pages, {len(test)} test pages; {new} images downloaded now, "
      f"{len(glob.glob(os.path.join(images_dir, '*.jpg')))} in {images_dir}")
if missing:
    hits = find_in_bucket(missing[0] + ".jpg", most=1)
    where = f" It is at {hits[0]!r}, so the images folder is somewhere else." if hits else ""
    print(f"WARNING: {len(missing)} images aren't at {prefix}images/<page>.jpg, e.g. {missing[0]}.{where}")

## Step 3. Prepare the pages for training

This decides how each page is read (whole, or a survey spread as two halves) and pairs every
training image with its text. It takes a few minutes, mostly finding the fold in the survey
spreads.

**Read the table it prints.** Every collection should have training, dev and test pages. If a
collection shows 0 training pages, `train.csv` isn't the full competition data.

In [ ]:
import prepare_data

items = prepare_data.build(DATA, "dev_ids.csv")

## Step 4. Check a survey spread by eye

The fold is found automatically, as the strip in the middle with the least writing. This shows a
spread with a **red line where it was cut**, then the two halves, with the start of the text each
half is trained on. The red line should run down the fold, between the two pages, not through
writing. Try a few spreads by changing `SPREAD`.

In [ ]:
from IPython.display import display
from PIL import Image, ImageDraw

parts = pd.read_csv(os.path.join(DATA, "parts.csv"), keep_default_na=False)
halves = parts[parts.part != "page"]
if halves.empty:
    print("No survey spreads were cut.")
else:
    SPREAD = halves.page_id.iloc[0]   # or type a page name, e.g. "survey_002_p0010"
    rows = parts[parts.page_id == SPREAD]
    img = Image.open(os.path.join(DATA, rows.image.iloc[0])).convert("RGB")
    left = [int(v) for v in rows[rows.part == "left"].box.iloc[0].split(",")]
    right = [int(v) for v in rows[rows.part == "right"].box.iloc[0].split(",")]
    fold = (left[2] + right[0]) // 2
    marked = img.copy()
    ImageDraw.Draw(marked).line([(fold, 0), (fold, img.height)], fill="red", width=max(3, img.width // 300))
    marked.thumbnail((900, 700))
    print(SPREAD)
    display(marked)
    texts = items[items.page_id == SPREAD]
    for name, b in (("left", left), ("right", right)):
        half = img.crop(b)
        half.thumbnail((450, 600))
        print(f"{name} half")
        display(half)
        match = texts[texts.box == ",".join(map(str, b))]
        if len(match):
            print("  trained on:", match.text.iloc[0][:120].replace("\n", " | "), "...")
    if SPREAD in set(items.page_id) and texts.box.eq("").all():
        print("This spread's label doesn't split at a page number, so it's trained on whole.")

**Done.** Go on to **Notebook 2** (`2_finetune.ipynb`).

To save the prepared data for teammates, copy the working folder to your bucket in a cell:
`!aws s3 sync {DATA} s3://<your bucket>/churro-work/data --quiet`